# 🎯 Reinforcement Fine-Tuning — o4-mini with Foundry SDK
## Microsoft Build 2026 | LAB231 — Phase 3

---

### The Idea

> *"SFT teaches by imitation. RFT teaches by exploration — the model tries things,
> gets a score from a grader, and learns from its own mistakes."*

**Reinforcement Fine-Tuning (RFT)** with Azure AI Foundry:
1. Give the model a task + tools
2. Let it attempt scenarios (calling tools, generating responses)
3. A **grader** scores each attempt (our custom Python grader as reward signal)
4. The model learns to maximize the reward

### Why o4-mini?
- It's a **reasoning model** — can reflect on mistakes and adjust strategy
- Already strong at 71% retail_quality — room to improve with targeted training — room to improve with targeted training
- Cost-effective for production deployment

### What you'll see:

| # | Section | What we do |
|---|---------|------------|
| 1 | Training Data | Load scenarios with expected outcomes |
| 2 | Upload to Foundry | Push training + validation datasets |
| 3 | The Grader | Custom reward signal (with tool scoring!) |
| 4 | Tool Configuration | 6 tools the model can call during training |
| 5 | Submit RFT Job | One API call — Foundry handles the rest |
| 6 | Monitor & Results | Track progress and view training metrics |

> 💡 **Speaker notes:** [`docs/3_rft_foundry.md`](../docs/3_rft_foundry.md)

---

## 1. Setup & Configuration

In [2]:
import os
import json
import time
from datetime import datetime
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
TOOL_URL = os.getenv("TOOL_URL")  # e.g. https://retail-tools-omkarm.azurewebsites.net/api

if not PROJECT_ENDPOINT:
    raise ValueError("PROJECT_ENDPOINT not set in .env")
if not TOOL_URL:
    raise ValueError("TOOL_URL not set in .env (your Function App URL)")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project_client.get_openai_client()

print(f"✅ Connected to Azure AI Foundry")
print(f"   Project: {PROJECT_ENDPOINT}")
print(f"   Tool URL: {TOOL_URL}")

✅ Connected to Azure AI Foundry
   Project: https://ai-account-44mf5lkxqssxm.services.ai.azure.com/api/projects/ai-project-omi-build26-azd-env
   Tool URL: https://retail-tools-omkarm.azurewebsites.net


---

## 2. Load Training & Validation Data

RFT needs:
- **Training scenarios** — the model will attempt these during training
- **Validation scenarios** — held-out set to measure generalization

Each scenario contains:
- `messages` — the conversation (system prompt + user request)
- `expected_resolution` — what the correct outcome looks like
- `expected_actions` — which tools should be called
- `expected_amounts` — correct financial calculations

The model doesn't see the "expected" fields — those are for the **grader** to score against.

In [3]:
train_path = "../data/retail_train.jsonl"
val_path = "../data/retail_val.jsonl"

with open(train_path) as f:
    train_data = [json.loads(line) for line in f]

with open(val_path) as f:
    val_data = [json.loads(line) for line in f]

print(f"📚 Training data: {len(train_data)} scenarios")
print(f"📏 Validation data: {len(val_data)} scenarios")

# Preview a sample
sample = train_data[0]
print(f"\n--- Sample scenario ---")
print(f"User: {sample['messages'][1]['content'][:100]}...")
print(f"Expected: {sample['expected_resolution'][:80]}...")

📚 Training data: 192 scenarios
📏 Validation data: 62 scenarios

--- Sample scenario ---
User: Yusuf Rossi. Need to exchange the Canvas Tote Bag from ORD-012 for a different size....
Expected: Action: exchange for LI-016 (reason: exchange). Amount: $0.00....


---

## 3. Upload Datasets to Azure AI Foundry

In [4]:
print("Uploading training dataset...")
train_file = client.files.create(
    file=open(train_path, "rb"),
    purpose="fine-tune"
)
print(f"✅ Training file: {train_file.id}")

print("Uploading validation dataset...")
val_file = client.files.create(
    file=open(val_path, "rb"),
    purpose="fine-tune"
)
print(f"✅ Validation file: {val_file.id}")

print(f"\n📊 Ready for RFT submission")
print(f"   Training: {len(train_data)} scenarios")
print(f"   Validation: {len(val_data)} scenarios")

Uploading training dataset...
✅ Training file: file-728fc8b2f8d34e16b4440f47a25968ae
Uploading validation dataset...
✅ Validation file: file-c33fe59e9ef44efaa75b2f1db1b68cd4

📊 Ready for RFT submission
   Training: 192 scenarios
   Validation: 62 scenarios


---

## 4. The RFT Grader — Reward Signal

This is the **key difference from SFT**: instead of showing the model correct answers,
we give it a **scoring function** (grader) that evaluates its attempts.

The RFT grader ([`scripts/retail_grader_rft_tools.py`](../scripts/retail_grader_rft_tools.py)) scores on 4 dimensions:

| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Action/Decision | 35% | Correct action (refund/exchange/deny)? |
| Financial Accuracy | 25% | Correct amounts (including $0.00)? |
| Tool Coverage | 25% | Called the right tools? |
| Workflow Order | 15% | Correct tool-call sequence? |

> 🔑 *"The grader is the reward signal. The model explores → gets scored → adjusts strategy.
> Extra/repeated tool calls are NOT penalized — exploration is encouraged."*

In [5]:
# Load the RFT grader (with tool scoring)
with open("../scripts/retail_grader_rft_tools.py") as f:
    GRADER_SOURCE = f.read()

print(f"✅ RFT Grader loaded ({len(GRADER_SOURCE)} chars)")
print(f"   Scoring: 35% action + 25% amounts + 25% tool coverage + 15% workflow")
print(f"   Pass threshold: 0.80")

✅ RFT Grader loaded (11638 chars)
   Scoring: 35% action + 25% amounts + 25% tool coverage + 15% workflow
   Pass threshold: 0.80


---

## 5. Configure Tool Access

During RFT training, the model can **actually call tools** — it's not just learning from static examples.
This is what makes RFT so powerful for agent tasks.

The 6 retail tools are served by our Function App:

| # | Tool | Purpose |
|---|------|--------|
| 1 | `get_order_details` | Retrieve order info, customer tier |
| 2 | `get_fulfillment_status` | Check delivery status |
| 3 | `check_resolution_policy` | Verify eligibility |
| 4 | `check_inventory` | Check stock for exchanges |
| 5 | `calculate_resolution` | Compute refund/credit amounts |
| 6 | `submit_resolution` | Execute the resolution |

> 🌐 **Live tools:** [retail-tools-omkarm.azurewebsites.net/demo](https://retail-tools-omkarm.azurewebsites.net/demo)

In [6]:
# Configure tools — each points to the Function App endpoint
TOOL_CONFIG = [
    {"name": "get_order_details", "server_url": f"{TOOL_URL}/tool/get_order_details", "headers": {}},
    {"name": "get_fulfillment_status", "server_url": f"{TOOL_URL}/tool/get_fulfillment_status", "headers": {}},
    {"name": "check_resolution_policy", "server_url": f"{TOOL_URL}/tool/check_resolution_policy", "headers": {}},
    {"name": "check_inventory", "server_url": f"{TOOL_URL}/tool/check_inventory", "headers": {}},
    {"name": "calculate_resolution", "server_url": f"{TOOL_URL}/tool/calculate_resolution", "headers": {}},
    {"name": "submit_resolution", "server_url": f"{TOOL_URL}/tool/submit_resolution", "headers": {}}
]

print(f"✅ {len(TOOL_CONFIG)} tools configured")
print(f"   Base URL: {TOOL_URL}/tool/{{tool_name}}")
for t in TOOL_CONFIG:
    print(f"   • {t['name']}")

✅ 6 tools configured
   Base URL: https://retail-tools-omkarm.azurewebsites.net/tool/{tool_name}
   • get_order_details
   • get_fulfillment_status
   • check_resolution_policy
   • check_inventory
   • calculate_resolution
   • submit_resolution


---

## 6. Submit the RFT Job 🚀

Key parameters:

| Parameter | Value | Why |
|-----------|-------|-----|
| `model` | o4-mini | Reasoning model, ideal for agent tasks |
| `method` | reinforcement | Not supervised — learns by exploration |
| `pass_threshold` | 0.80 | Grader score ≥ 80% = success |
| `max_episode_steps` | 5 | Max tool-call rounds per scenario |
| `n_epochs` | 3 | Training iterations |
| `reasoning_effort` | medium | Balance between thinking and speed |

> ⏱️ Training takes **2-4 hours**. The job is pre-completed for this demo.

In [7]:
# Submit RFT job
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
suffix = f"retail-rft-{timestamp}"

print(f"🚀 Submitting RFT job...")
print(f"   Model: o4-mini")
print(f"   Suffix: {suffix}")
print()

job = client.fine_tuning.jobs.create(
    model="o4-mini",
    training_file=train_file.id,
    validation_file=val_file.id,
    suffix=suffix,
    method={
        "type": "reinforcement",
        "reinforcement": {
            "grader": {
                "type": "python",
                "name": "retail_quality",
                "source": GRADER_SOURCE.strip(),
                "pass_threshold": 0.80,
            },
            "tools": TOOL_CONFIG,
            "max_episode_steps": 5,
            "hyperparameters": {
                "n_epochs": 3,
                "learning_rate_multiplier": 1.0,
                "compute_multiplier": 1.5,
                "reasoning_effort": "medium",
                "eval_interval": 5,
                "eval_samples": 10,
            },
        }
    },
)

JOB_ID = job.id
print("━" * 55)
print("✅ RFT JOB SUBMITTED SUCCESSFULLY!")
print("━" * 55)
print(f"\n  Job ID: {JOB_ID}")
print(f"  Status: {job.status}")
print(f"  Model: {job.model}")
print(f"\n  ⏰ Expected completion: 2-4 hours")
print(f"\n  💡 Monitor in Azure AI Foundry → Fine-tuning → Jobs")

🚀 Submitting RFT job...
   Model: o4-mini
   Suffix: retail-rft-20260531-054036

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ RFT JOB SUBMITTED SUCCESSFULLY!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Job ID: ftjob-19ac9e7d9a404fc7b9b37b0e4cb9163a
  Status: pending
  Model: o4-mini-2025-04-16

  ⏰ Expected completion: 2-4 hours

  💡 Monitor in Azure AI Foundry → Fine-tuning → Jobs


---

## 7. Monitor Job Progress

Check the status of your RFT job. You can also monitor in
[Azure AI Foundry → Fine-tuning](https://ai.azure.com).

In [ ]:
# Check job status
job_status = client.fine_tuning.jobs.retrieve(JOB_ID)

print(f"Job: {job_status.id}")
print(f"Status: {job_status.status}")
print(f"Model: {job_status.model}")
print(f"Created: {job_status.created_at}")

if job_status.status == "succeeded":
    print(f"\n🎉 JOB COMPLETED!")
    print(f"   Fine-tuned model: {job_status.fine_tuned_model}")
    print(f"\n   Next: Deploy as agent and run evaluation")
elif job_status.status == "failed":
    print(f"\n❌ JOB FAILED: {job_status.error}")
else:
    print(f"\n⏳ Job is {job_status.status}...")
    print(f"   Check back in 30-60 minutes")

In [ ]:
# View training events (optional)
events = client.fine_tuning.jobs.list_events(JOB_ID, limit=20)

print(f"Recent training events:\n")
for event in events.data:
    ts = datetime.fromtimestamp(event.created_at)
    print(f"[{ts}] {event.message}")

---

## 📊 Results

After RFT training, the fine-tuned o4-mini model is deployed as `o4-mini-finetuned`.

### Performance:

| Model | retail_quality | Method |
|-------|:---:|--------|
| o4-mini (base) | 71.0% | — |
| o4-mini (after RFT) | **82.3%** ⬆️ | Reinforcement |
| GPT-5.4 (teacher) | 64.5% | — |

### Why RFT works better than SFT here:

| SFT | RFT |
|-----|-----|
| Learns by imitation | Learns by exploration |
| Limited to teacher's patterns | Discovers new strategies |
| Can't exceed teacher quality | Can surpass teacher |
| Works for any model | Best with reasoning models |

---

### Deploy & test:

```bash
# Deploy the RFT-trained agent
cd deploy && azd deploy retail-o4-mini-finetuned

# Test it
azd ai agent invoke --agent-name retail-o4-mini-finetuned \
  --message "Noah Brown. I changed my mind on the Bluetooth Speaker from ORD-010, can I return it?"
```

---

**Next → [`4_rft_lowlevel.ipynb`](./4_rft_lowlevel.ipynb)** — Low-level RFT on Qwen3-32B with tinker APIs for maximum control.